# TASK 03 — Vietnamese Used Household Appliance Pricing Dataset Audit

**Project:** SecondLife AI Pricing  
**Domain:** Pre-owned Household Appliances  
**Country / Currency:** Vietnam (VND)  
**Target Architecture:** Multimodal Feature Extraction (PhoBERT Text + CLIP Vision + Tabular Features + Regression Head)  
**Status:** **DATASET ACQUISITION BLOCKED** (Public Sources Inadequate / Unauthorized)  

> **IMPORTANT DISCLAIMER:** This notebook demonstrates the canonical schema, validation engine, Vietnamese text normalization, and group-aware leakage-free splitting. It executes audits on a strictly labeled synthetic benchmark dataset (`synthetic_test_only`). It does **NOT** train a price prediction regression model, does **NOT** convert Brazilian retail BRL prices to VND, and does **NOT** scrape unauthorized marketplace data.

## 1. Environment & Dependencies Setup

We verify Python dependencies: PyTorch, Pandas, Pydantic v2, Scikit-learn, Seaborn, and Matplotlib.

In [ ]:
import os
import sys
import json
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import SecondLife pricing data pipeline modules
from src.data.pricing_schema import (
    ApplianceCategory,
    ConditionTier,
    ListingSource,
    LicenseStatus,
    PricingListing,
    create_empty_schema_dataframe,
)
from src.data.pricing_validation import PricingValidator, ValidationReport
from src.data.pricing_preprocessing import (
    CategoryNormalizer,
    VietnameseTextNormalizer,
    PriceNormalizer,
    DataSplitter,
)

print(f"[INFO] Python {sys.version.split()[0]} | Pandas {pd.__version__}")
print("[INFO] Pricing modules successfully loaded.")

## 2. Public Dataset Research & Acquisition Assessment

An exhaustive review of Hugging Face, Kaggle, and academic repositories confirms that **no legitimate public dataset pairs authentic photos of used appliances with their secondary market listing prices in VND**.

| Candidate Source | Stated Content | Why Unsuitable / Blocked |
|:---|:---|:---|
| `TudoBonus/tudobonus` | 109K rows, 14K images | Brazilian retail prices in BRL; South American appliances; foreign exchange distortion. |
| `slconnin/amazon_appliance_prices` | Amazon US reviews | English text, USD retail prices, zero images. |
| `5CD-AI/Vietnamese-Ecommerce-Alpaca` | E-commerce instruction text | Synthetic/unstructured text; no listing prices or paired photos. |
| `tnhyen/vietnamese-ecommerce-ner` | ViEcomNER token annotations | NLP token classification only; zero images or pricing. |
| `Shopee / Tiki Reviews` | Product reviews | Sentiment analysis on new retail goods; no second-hand condition degradation. |
| `Chợ Tốt Web Scraping` | Raw marketplace ads | **BLOCKED:** Automated scraping violates Terms of Service and data privacy laws. |

**Conclusion:** Dataset acquisition from public sources is **BLOCKED**. A formal data acquisition strategy (partner data integration and native appraisal loop) is outlined at the end of this notebook.

## 3. Canonical Schema & Pydantic Validation

The `PricingListing` model validates required fields, field boundaries (price in [50K - 150M VND], age in [0 - 30 years]), uppercase brand normalization, and license compliance.

In [ ]:
# Demonstrate valid listing record creation
example_record = {
    "listing_id": "SLVN-2026-DEMO01",
    "title": "Tủ lạnh Panasonic Inverter 255 lít còn mới 95%",
    "description": "Tủ lạnh chính hãng mua điện máy xanh, chạy êm làm lạnh sâu.",
    "category": ApplianceCategory.REFRIGERATOR,
    "brand": "panasonic",  # Will be auto-normalized to uppercase
    "model_number": "NR-BV280QSVN",
    "condition": ConditionTier.EXCELLENT,
    "age_years": 1.5,
    "image_paths": ["data/clip_sample/3356319/1656689035.jpg"],
    "listing_price_vnd": 4500000.0,
    "listed_at": "2026-10-14T08:30:00Z",
    "source": ListingSource.SYNTHETIC_BENCHMARK,
    "source_record_id": "MOCK-001",
    "license_status": LicenseStatus.SYNTHETIC_TEST_ONLY,
}

listing = PricingListing(**example_record)
print("[INFO] Valid Listing Successfully Parsed:")
print(f"  - ID: {listing.listing_id}")
print(f"  - Brand: {listing.brand}")
print(f"  - Category: {listing.category.value}")
print(f"  - Condition: {listing.condition.value}")
print(f"  - Price: {listing.listing_price_vnd:,.0f} VND")
print(f"  - License: {listing.license_status.value}")

## 4. Vietnamese Text & Category Normalization

We test unicode NFC normalization, marketplace abbreviation expansion, and multilingual category mapping.

In [ ]:
test_phrases = [
    "  Tủ   lạnh  inv bh 12 tháng !!!  ",
    "Máy giặt sấy cửa ngang 9kg giá 6tr5",
    "Điều hòa Daikin 1.5 HP inverter mới 99%",
]

print("=== VIETNAMESE TEXT NORMALIZATION ===")
for text in test_phrases:
    norm = VietnameseTextNormalizer.normalize_text(text, expand_abbreviations=True)
    print(f"Original:   {text}")
    print(f"Normalized: {norm}\n")

print("=== MULTILINGUAL CATEGORY MAPPING ===")
test_categories = [
    "Máy lạnh", "Tủ lạnh", "Lò vi sóng", "Máy giặt sấy", 
    "Ar Condicionado", "Geladeira", "Lavadora de roupas"
]
for raw_cat in test_categories:
    canonical = CategoryNormalizer.normalize(raw_cat)
    print(f"{raw_cat:20} -> {canonical.value if canonical else 'None'}")

## 5. Audit on Synthetic Benchmark Dataset

We create a realistic synthetic benchmark covering major household appliance categories with intentional edge cases (boundary violations, IQR price outliers, exact duplicates) to demonstrate the automated `PricingValidator` engine.

In [ ]:
# Generate realistic synthetic benchmark data
np.random.seed(42)

benchmark_records = []
categories_pool = [
    ("Refrigerator", "PANASONIC", ["NR-BV280", "NR-BL300"], (2_500_000, 7_000_000)),
    ("Washing Machine", "ELECTROLUX", ["EWF8025", "EWF9024"], (3_000_000, 8_500_000)),
    ("Air Conditioner", "DAIKIN", ["FTKB25XVMV", "FTKB35XVMV"], (4_000_000, 10_000_000)),
    ("Microwave", "SHARP", ["R-205VN", "R-G222VN"], (500_000, 1_800_000)),
    ("Dishwasher", "BOSCH", ["SMS46GI01P", "SMS6ZCI42E"], (8_000_000, 22_000_000)),
]
conditions_pool = ["Like New", "Excellent", "Good", "Fair", "Poor"]

idx = 1
for cat, brand, models, (min_p, max_p) in categories_pool:
    for i in range(10):
        price = float(np.random.randint(min_p // 100_000, max_p // 100_000) * 100_000)
        benchmark_records.append({
            "listing_id": f"BENCH-{idx:04d}",
            "title": f"{cat} {brand} {models[i % 2]} nguyên bản dùng tốt",
            "description": f"Thanh lý {cat.lower()} {brand} gia đình đang sử dụng tốt",
            "category": cat,
            "brand": brand,
            "model_number": models[i % 2],
            "condition": conditions_pool[i % len(conditions_pool)],
            "age_years": round(float(np.random.uniform(0.5, 6.0)), 1),
            "image_paths": json.dumps([f"sample_images/{idx}.jpg"]),
            "listing_price_vnd": price,
            "listed_at": "2026-10-10T12:00:00Z",
            "source": "synthetic_benchmark",
            "source_record_id": f"SRC-{idx:04d}",
            "license_status": "synthetic_test_only",
        })
        idx += 1

# Insert controlled anomaly: Extreme Outlier (95M Refrigerator)
benchmark_records.append({
    "listing_id": f"BENCH-{idx:04d}",
    "title": "Tủ lạnh mini giá gõ nhầm 95 triệu",
    "description": "Lỗi nhập liệu thừa số 0",
    "category": "Refrigerator",
    "brand": "PANASONIC",
    "model_number": "NR-MINI",
    "condition": "Poor",
    "age_years": 8.0,
    "image_paths": json.dumps(["sample_images/outlier.jpg"]),
    "listing_price_vnd": 95_000_000.0,  # Extreme outlier
    "listed_at": "2026-10-11T12:00:00Z",
    "source": "synthetic_benchmark",
    "source_record_id": f"SRC-{idx:04d}",
    "license_status": "synthetic_test_only",
})
idx += 1

# Insert controlled anomaly: Exact Duplicate
dup_record = benchmark_records[0].copy()
dup_record["listing_id"] = f"BENCH-{idx:04d}"
benchmark_records.append(dup_record)

df_benchmark = pd.DataFrame(benchmark_records)
print(f"[INFO] Generated synthetic benchmark with {len(df_benchmark)} listings across {df_benchmark['category'].nunique()} categories.")
df_benchmark.head(3)

## 6. Execution of Automated Validation Engine

The `PricingValidator` assesses schema conformity, detects price outliers using category-specific IQR thresholds, and checks for duplicates.

In [ ]:
validator = PricingValidator(iqr_multiplier=1.5)
report, df_audited = validator.validate_dataframe(df_benchmark)

print(report.to_markdown())

## 7. Price Distribution & Outlier Visualization

Boxplot and scatter visualization displaying category price distributions and the flagged outlier.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_audited[df_audited['listing_price_vnd'] < 50_000_000], x="category", y="listing_price_vnd", palette="Set2")
plt.title("Synthetic Benchmark: Secondary Market Appliance Price Distribution in Vietnam (VND)")
plt.xlabel("Appliance Category")
plt.ylabel("Price (VND)")
plt.xticks(rotation=30, ha="right")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f"{int(x):,} đ"))
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("Flagged Outlier Records:")
df_outliers = df_audited[df_audited['is_price_outlier']][['listing_id', 'category', 'brand', 'listing_price_vnd', 'outlier_reason']]
display(df_outliers)

## 8. Group-Aware Train/Validation/Test Split (Zero Leakage)

To prevent multi-view or re-listed listings of the same product instance from leaking across training and evaluation splits, `DataSplitter` partitions clusters into disjoint sets.

In [ ]:
df_train, df_val, df_test = DataSplitter.split_train_val_test(
    df_audited,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    random_state=42
)

train_groups = set(df_train["product_group_id"].unique())
val_groups = set(df_val["product_group_id"].unique())
test_groups = set(df_test["product_group_id"].unique())

print(f"Train set:      {len(df_train)} rows ({len(train_groups)} product groups)")
print(f"Validation set: {len(df_val)} rows ({len(val_groups)} product groups)")
print(f"Test set:       {len(df_test)} rows ({len(test_groups)} product groups)")

# Verify zero group leakage
assert train_groups.isdisjoint(val_groups), "Leakage between Train and Val!"
assert train_groups.isdisjoint(test_groups), "Leakage between Train and Test!"
assert val_groups.isdisjoint(test_groups), "Leakage between Val and Test!"
print("\n[PASSED] Zero-leakage assertion verified across all splits!")

## 9. Data Acquisition Action Plan

Because public datasets with real Vietnamese appliance photos and VND prices do not exist, the SecondLife project will execute the following compliant acquisition steps:
1. **Execute Refurbisher Partner Agreements:** Direct data ingestion with warranty-backed refurbishers (Điện Máy Cũ) in HCMC and Hanoi.
2. **Platform Appraisal Collection:** Ingest user-submitted appraisal listings (multi-view photos, descriptions, brand/model) and log final closed transactions.
3. **Multimodal Fusion Pipeline:** Once real partner data is ingested, freeze CLIP for vision embeddings, freeze PhoBERT for textual title/description embeddings, and train a CatBoost / MLP regression head for VND price prediction.